# Phase 1 — From API data to my own database

**Goal.** Download raw OpenStreetMap data for Prabhag 11 (Bopodi – Savitribai Phule Pune University) plus a 2 km margin,
clean it, and load it into PostGIS in EPSG:4326 with a UTM 43N copy for metric work.

What the earlier PuneStay project did: fetched points from a ready API / static file and measured straight-line distance.
What this notebook does instead: pulls the *raw* OSM objects (services, building footprints, walking network) with OSMnx,
cleans them myself, and owns the data in PostGIS.

**Run order:** `docker compose up -d` first (the database must be healthy), then run every cell top to bottom.
Downloads are cached in `data/raw/osmnx_cache/` (git-ignored), so a re-run is fast.

| Step | Output table | CRS |
|---|---|---|
| area polygon | `area` | 4326 (+ `geom_utm` 32643) |
| services (PGs and hostels) as points | `services` | 4326 (+ UTM) |
| building footprints | `buildings` | 4326 (+ UTM) |
| walking network | `walk_nodes`, `walk_edges` | 4326 (+ UTM) |


In [ ]:
import json, math, re, time, sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import osmnx as ox
import requests
import shapely
from shapely.geometry import Point
from sqlalchemy import text

# repo root works whether the kernel starts in /analysis or in the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT / "analysis"))
from dbconf import engine_from_env

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = str(RAW / "osmnx_cache")
ox.settings.log_console = False
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

UTM = 32643          # WGS 84 / UTM zone 43N (Pune, 73.8 E -> zone floor((73.8+180)/6)+1 = 43)
MARGIN_M = 2000      # the brief: area plus a 2 km margin
print("osmnx", ox.__version__, "| geopandas", gpd.__version__)

## 1. Study-area polygon and the download extent (area + 2 km)

In [ ]:
area = gpd.read_file(ROOT / "data" / "area" / "prabhag11.geojson")
area = area.set_crs(4326, allow_override=True)
area_utm = area.to_crs(UTM)
print("area km2 (UTM 43N):", round(float(area_utm.area.iloc[0]) / 1e6, 3))

# buffer in METRES (UTM), then back to degrees for the Overpass query
extent_utm = area_utm.geometry.buffer(MARGIN_M)
extent = gpd.GeoSeries(extent_utm, crs=UTM).to_crs(4326).iloc[0]
print("download extent km2:", round(float(extent_utm.area.iloc[0]) / 1e6, 1))
print("extent bbox (W,S,E,N):", [round(v, 4) for v in extent.bounds])

## 2. Services: PGs and hostels

OSM has no single tag for an Indian PG, so I union several:
`tourism=hostel`, `tourism=guest_house`, `building=dormitory`, `amenity=dormitory`, and any object whose **name**
says *PG*, *Paying Guest* or *Hostel* (university hostels are usually `building=yes` + a hostel name).
OSMnx cannot filter by a name regex, so the name search goes straight to the Overpass API with the same extent.

In [ ]:
SERVICE_TAGS = {"tourism": ["hostel", "guest_house"], "building": ["dormitory"], "amenity": ["dormitory"]}
svc_tagged = ox.features_from_polygon(extent, SERVICE_TAGS)
print("tag-based raw features:", len(svc_tagged))
svc_tagged = svc_tagged.reset_index()          # element_type / osmid become columns

# --- name-based search (regex) via Overpass; generic User-Agent, no personal data ---
ring = extent.exterior.coords if extent.geom_type == "Polygon" else max(extent.geoms, key=lambda g: g.area).exterior.coords
poly = " ".join(f"{lat:.6f} {lon:.6f}" for lon, lat in ring)
NAME_RE = "[Hh]ostel|[Pp]aying [Gg]uest|(^|[^A-Za-z])PG([^A-Za-z]|$)"
query = f'[out:json][timeout:180];nwr["name"~"{NAME_RE}"](poly:"{poly}");out center tags;'
(ROOT / "sql").mkdir(exist_ok=True)
(ROOT / "analysis" / "overpass_name_query.txt").write_text(query, encoding="utf-8")

def overpass(query, tries=6):
    mirrors = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
    for i in range(tries):
        url = mirrors[i % len(mirrors)]
        try:
            r = requests.post(url, data={"data": query}, headers={"User-Agent": "pune-pg-webgis/0.1"}, timeout=200)
            if r.status_code == 200:
                return r.json()["elements"]
            print("  ", url, "HTTP", r.status_code)
        except requests.RequestException as e:
            print("  ", url, type(e).__name__)
        time.sleep(20)
    raise RuntimeError("Overpass did not answer")

rows = []
for e in overpass(query):
    lat, lon = (e["lat"], e["lon"]) if "lat" in e else (e["center"]["lat"], e["center"]["lon"])
    t = e.get("tags", {})
    rows.append({"element_type": e["type"], "osmid": e["id"], "name": t.get("name"), "tourism": t.get("tourism"),
                 "building": t.get("building"), "amenity": t.get("amenity"), "highway": t.get("highway"),
                 "geometry": Point(lon, lat)})
svc_named = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326)
print("name-based raw features:", len(svc_named))

### Cleaning rules
1. polygons/buildings → one **point** each (`representative_point`, which always lies inside the outline);
2. drop objects that only *carry* the word hostel: roads (`highway=*`) and canteens (`amenity=fast_food|restaurant|cafe`);
3. drop **duplicates**: same normalised name within 30 m (a node and a building outline for the same hostel);
4. keep `name`, the type tags and the OSM id, and a `service_type` label.

In [ ]:
def to_points(gdf):
    g = gdf.copy()
    g["geometry"] = g.geometry.apply(lambda geom: geom if geom.geom_type == "Point" else geom.representative_point())
    return g

keep_cols = ["element_type", "osmid", "name", "tourism", "building", "amenity", "highway", "geometry"]
for c in keep_cols:
    if c not in svc_tagged.columns:
        svc_tagged[c] = None
raw = pd.concat([to_points(svc_tagged[keep_cols]), svc_named[keep_cols]], ignore_index=True)
raw = gpd.GeoDataFrame(raw, geometry="geometry", crs=4326)
raw = raw.drop_duplicates(subset=["element_type", "osmid"])
print("raw combined:", len(raw))

# rule 2: not a place to stay
is_road = raw["highway"].notna()
is_food = raw["amenity"].isin(["fast_food", "restaurant", "cafe"])
dropped = raw[is_road | is_food]
svc = raw[~(is_road | is_food)].copy()
print("dropped roads/canteens:", len(dropped), list(dropped["name"]))

# rule 3: duplicates = same normalised name within 30 m
def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower()) if pd.notna(s) else ""
svc["name_norm"] = svc["name"].map(norm)
svc_utm = svc.to_crs(UTM)
keep = []
for i, row in svc_utm.iterrows():
    dup = False
    for j in keep:
        other = svc_utm.loc[j]
        if row["name_norm"] and row["name_norm"] == other["name_norm"] and row.geometry.distance(other.geometry) < 30:
            dup = True
            break
    if not dup:
        keep.append(i)
svc = svc.loc[keep].copy()
print("after de-duplication:", len(svc))

# label
def service_type(r):
    if r["tourism"] == "guest_house": return "guest_house"
    if r["tourism"] == "hostel" or r["building"] == "dormitory" or r["amenity"] == "dormitory": return "hostel"
    n = str(r["name"]).lower()
    if "paying" in n or re.search(r"(^|[^a-z])pg([^a-z]|$)", n): return "pg"
    return "hostel"
svc["service_type"] = svc.apply(service_type, axis=1)
svc["inside_area"] = svc.within(area.geometry.iloc[0])
svc = svc.drop(columns=["name_norm", "highway"]).reset_index(drop=True)
svc.insert(0, "service_id", range(1, len(svc) + 1))
print(svc["service_type"].value_counts().to_dict(), "| inside Prabhag 11:", int(svc["inside_area"].sum()),
      "| unnamed inside:", int((svc["inside_area"] & svc["name"].isna()).sum()))
svc[svc["inside_area"]][["service_id", "name", "service_type", "element_type", "osmid"]]

## 3. Building footprints (area + 2 km)

In [ ]:
t0 = time.time()
bld = ox.features_from_polygon(extent, {"building": True}).reset_index()
bld = bld[bld.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
cols = {"element_type": "element_type", "osmid": "osmid", "building": "building", "name": "name",
        "building:levels": "levels", "addr:street": "street", "geometry": "geometry"}
for c in cols:
    if c not in bld.columns:
        bld[c] = None
bld = bld[list(cols)].rename(columns=cols)
bld.insert(0, "building_id", range(1, len(bld) + 1))
bld["area_m2"] = bld.to_crs(UTM).area.round(1)
bld["inside_area"] = bld.representative_point().within(area.geometry.iloc[0])
print(f"buildings: {len(bld):,} (inside Prabhag 11: {int(bld['inside_area'].sum()):,}) in {time.time()-t0:.0f} s")
bld["building"].value_counts().head(8)

## 4. Walking network (area + 2 km)

`network_type="walk"` keeps every way a pedestrian may use (footways, paths, steps, residential and service roads,
living streets) and drops motorways and trunk roads with `foot=no`. The graph is **undirected for walking** (one-way
restrictions apply to cars, not pedestrians) and simplified so each edge is a street segment between two junctions.

In [ ]:
t0 = time.time()
G = ox.graph_from_polygon(extent, network_type="walk", simplify=True, retain_all=False)
ox.save_graphml(G, RAW / "walk_extent.graphml")       # reused in Phase 3 (git-ignored)
nodes, edges = ox.graph_to_gdfs(G)
nodes = nodes.reset_index()[["osmid", "x", "y", "street_count", "geometry"]].rename(columns={"osmid": "node_id"})
edges = edges.reset_index()
for c in ["name", "highway", "length", "oneway"]:
    if c not in edges.columns:
        edges[c] = None
edges = edges[["u", "v", "key", "osmid", "name", "highway", "length", "geometry"]].copy()
# OSMnx stores list values when a simplified edge merges several ways; make them text for PostGIS
for c in ["osmid", "name", "highway"]:
    edges[c] = edges[c].apply(lambda v: ";".join(map(str, v)) if isinstance(v, list) else v)
edges.insert(0, "edge_id", range(1, len(edges) + 1))
print(f"nodes {len(nodes):,}  edges {len(edges):,}  total length {edges['length'].sum()/1000:.1f} km  in {time.time()-t0:.0f} s")

## 5. Load into PostGIS (EPSG:4326) and add the UTM 43N copy

GeoPandas writes straight to PostGIS. All four layers go in as EPSG:4326 (the exchange CRS), then
`sql/01_utm_copies.sql` adds a `geom_utm` column in EPSG:32643 to each table for metric work.

In [ ]:
engine = engine_from_env()
area_out = area.copy()
area_out.insert(0, "area_id", 1)
area_out["area_km2_utm"] = round(float(area_utm.area.iloc[0]) / 1e6, 4)

with engine.begin() as con:
    area_out.to_postgis("area", con, if_exists="replace", index=False)
    svc.to_postgis("services", con, if_exists="replace", index=False)
    bld.to_postgis("buildings", con, if_exists="replace", index=False, chunksize=5000)
    nodes.to_postgis("walk_nodes", con, if_exists="replace", index=False, chunksize=10000)
    edges.to_postgis("walk_edges", con, if_exists="replace", index=False, chunksize=10000)
    # GeoPandas names the geometry column "geometry"; rename to the conventional "geom"
    for t in ["area", "services", "buildings", "walk_nodes", "walk_edges"]:
        con.execute(text(f'ALTER TABLE {t} RENAME COLUMN geometry TO geom'))
        # to_postgis also creates a GiST index on the geometry; drop it so Q2 can start from a clean slate
        con.execute(text(f'DROP INDEX IF EXISTS idx_{t}_geometry'))
    con.execute(text((ROOT / "sql" / "01_utm_copies.sql").read_text(encoding="utf-8")))
print("loaded")

In [ ]:
with engine.connect() as con:
    summary = pd.read_sql(text('''
        SELECT 'area' AS tbl, count(*) AS rows, ST_SRID(min(geom)) AS srid, ST_SRID(min(geom_utm)) AS srid_utm FROM area
        UNION ALL SELECT 'services', count(*), ST_SRID(min(geom)), ST_SRID(min(geom_utm)) FROM services
        UNION ALL SELECT 'buildings', count(*), ST_SRID(min(geom)), ST_SRID(min(geom_utm)) FROM buildings
        UNION ALL SELECT 'walk_nodes', count(*), ST_SRID(min(geom)), ST_SRID(min(geom_utm)) FROM walk_nodes
        UNION ALL SELECT 'walk_edges', count(*), ST_SRID(min(geom)), ST_SRID(min(geom_utm)) FROM walk_edges'''), con)
    idx = pd.read_sql(text("SELECT tablename, indexname FROM pg_indexes WHERE schemaname='public' ORDER BY 1,2"), con)
display(summary)
print("indexes now present (should be primary keys only, no GiST yet):")
display(idx)

## 6. Quick look

Services inside the ward (black) and in the margin (grey) over the building footprints.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 8))
bld.plot(ax=ax, color="#ddd", linewidth=0)
edges.plot(ax=ax, color="#bbb", linewidth=0.3)
area.boundary.plot(ax=ax, color="red", linewidth=1.5)
svc[~svc["inside_area"]].plot(ax=ax, color="grey", markersize=14)
svc[svc["inside_area"]].plot(ax=ax, color="black", markersize=18)
ax.set_title("Prabhag 11: PGs/hostels (black inside, grey in the 2 km margin), buildings, walking network")
ax.set_axis_off()
(ROOT / "docs" / "screenshots").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "docs" / "screenshots" / "phase1_overview.png", dpi=130, bbox_inches="tight")
plt.show()